## Pytorch Training Pipeline
1. Dataset Loading
2. Data Preprocessing
3. Define Model Architecture
4. Training Loop
   1. Create the model
   2. Forward pass
   3. Compute loss
   4. Backward pass
   5. Parameter update
5. Evaluation

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [11]:
class NeuralNetwork():
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.rand(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self, y_pred, y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        loss = -torch.mean(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))
        return loss
    
model = NeuralNetwork(X_train_tensor)

learning_rate = 0.1
epochs = 100

for epoch in range(epochs):
    y_pred = model.forward(X_train_tensor)
    loss = model.loss_function(y_pred, y_train_tensor)
    
    loss.backward()
    
    with torch.no_grad():   # Update weights and bias without tracking gradients as perameters are updated manually
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
        
        model.weights.grad.zero_()
        model.bias.grad.zero_()
    
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')

Epoch 1/100, Loss: 3.8996383926711906
Epoch 2/100, Loss: 3.7935492647186293
Epoch 3/100, Loss: 3.6846302226098873
Epoch 4/100, Loss: 3.5724338465148135
Epoch 5/100, Loss: 3.4567466057174525
Epoch 6/100, Loss: 3.3377890555679373
Epoch 7/100, Loss: 3.213824914352372
Epoch 8/100, Loss: 3.086348329621305
Epoch 9/100, Loss: 2.9506098106145826
Epoch 10/100, Loss: 2.8097274093163462
Epoch 11/100, Loss: 2.6644951827397385
Epoch 12/100, Loss: 2.5127235215523402
Epoch 13/100, Loss: 2.3558384044400325
Epoch 14/100, Loss: 2.197327991251631
Epoch 15/100, Loss: 2.0372539305550843
Epoch 16/100, Loss: 1.8810725688338132
Epoch 17/100, Loss: 1.731548468003468
Epoch 18/100, Loss: 1.5923159724750398
Epoch 19/100, Loss: 1.4647206378787434
Epoch 20/100, Loss: 1.350005281274704
Epoch 21/100, Loss: 1.249206271918307
Epoch 22/100, Loss: 1.1601436110843915
Epoch 23/100, Loss: 1.0816247478639813
Epoch 24/100, Loss: 1.0180087893532934
Epoch 25/100, Loss: 0.9676957553343357
Epoch 26/100, Loss: 0.9285387755602185
E

In [8]:
with torch.no_grad():
    y_pred_test = model.forward(X_test_tensor)
    y_pred_labels = (y_pred_test > 0.5).float()
    accuracy = (y_pred_labels.squeeze() == y_test_tensor).float().mean()
    print(f'Test Accuracy: {accuracy.item() * 100:.2f}%')

Test Accuracy: 54.39%
